In [1]:
!pip install -q polars huggingface_hub implicit faiss-cpu torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.0 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install scipy

In [3]:
import polars as pl
from datetime import timedelta, date, datetime
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import numpy as np
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import itertools
from huggingface_hub import snapshot_download
from scipy.sparse import coo_matrix

## Анализ таргета

In [4]:
snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/retail/**"]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

'/kaggle/working'

In [5]:

events_dir = "/kaggle/working/dataset/small/retail/events"

In [6]:
events_df = (
    pl.scan_parquet(f"{events_dir}/*.pq", include_file_paths="path")
    .with_columns(
        pl.col("path")
          .str.extract(r"(\d+)\.pq", group_index=1)
          .cast(pl.Int32)
          .alias("day")
    )
    .sort("day")
    .drop("path")
)

In [7]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [8]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 18h 7m 6s 19695µs,80440318,"""fmcg_956925""","""catalog""","""view""","""android""",1082
1082d 18h 7m 6s 291283µs,84851155,"""fmcg_396815""","""search""","""view""","""android""",1082
1082d 18h 7m 6s 296610µs,36586846,"""fmcg_579060""","""catalog""","""view""","""android""",1082
1082d 18h 7m 6s 606337µs,38365565,"""fmcg_993332""","""catalog""","""view""","""android""",1082
1082d 18h 7m 6s 649431µs,88459534,"""fmcg_878393""","""main""","""click""","""android""",1082


In [9]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""view""",253468410
"""order""",7433721
"""added_to_cart""",3852145
"""click""",2289696


Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [10]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""view""",1.053559,0.719574,1.02643
"""order""",35.923325,3.608843,5.993607
"""added_to_cart""",69.323448,4.253105,8.32607
"""click""",116.628571,4.767532,10.799471


In [11]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [12]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 18h 7m 6s 972425µs,89598392,"""fmcg_722977""","""other""","""order""","""android""",1082,35.923325,3.608843,5.993607,0,0,0,0
1082d 18h 7m 8s 244124µs,75137824,"""fmcg_262964""","""other""","""order""","""other""",1082,35.923325,3.608843,5.993607,0,0,0,0
1082d 18h 7m 9s 341841µs,14452431,"""fmcg_982039""","""other""","""order""","""other""",1082,35.923325,3.608843,5.993607,0,0,0,0
1082d 18h 7m 9s 594118µs,65266594,"""fmcg_180861""","""other""","""order""","""android""",1082,35.923325,3.608843,5.993607,0,0,0,0
1082d 18h 7m 19s 779819µs,60204165,"""fmcg_337385""","""other""","""order""","""other""",1082,35.923325,3.608843,5.993607,0,0,0,0


## Global Temporal Split 

In [13]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [14]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [15]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [16]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [17]:
train, test = global_temporal_split(events_df, 0.2)

In [18]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
84866206,[1268d 11h 34m 10s 742805µs],"[""fmcg_996496""]","[""catalog""]","[""click""]","[""android""]",[1268],[116.628571],[4.767532],[10.799471],[0],[0],[0],[1]
80565128,"[1279d 20h 15m 58s 645651µs, 1279d 23h 34m 27s 862611µs, … 1308d 16h 25m 38s 896090µs]","[""fmcg_1095351"", ""fmcg_710309"", … ""fmcg_991830""]","[""catalog"", ""catalog"", … ""catalog""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1279, 1279, … 1308]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
32992205,"[1308d 6h 54m 44s 336471µs, 1308d 23h 22m 20s 625983µs]","[""fmcg_708219"", ""fmcg_980512""]","[""main"", ""main""]","[""click"", ""click""]","[""android"", ""android""]","[1308, 1308]","[116.628571, 116.628571]","[4.767532, 4.767532]","[10.799471, 10.799471]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
7738980,"[1273d 10h 11m 9s 186914µs, 1273d 14h 29m 14s 205428µs, … 1287d 9h 49m 9s 297384µs]","[""fmcg_985601"", ""fmcg_744316"", … ""fmcg_1162172""]","[""catalog"", ""catalog"", … ""i2i""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1273, 1273, … 1287]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
57499221,"[1276d 10h 43m 52s 827672µs, 1276d 12h 20m 54s 245649µs, … 1308d 15h 54m 42s 768733µs]","[""fmcg_360238"", ""fmcg_333271"", … ""fmcg_550507""]","[""search"", ""search"", … ""search""]","[""click"", ""click"", … ""click""]","[""android"", ""ios"", … ""android""]","[1276, 1276, … 1308]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
54572163,"[1273d 20h 56m 7s 229479µs, 1273d 21h 7m 31s 863318µs, … 1289d 20h 16m 14s 509519µs]","[""fmcg_886708"", ""fmcg_716686"", … ""fmcg_942632""]","[""search"", ""search"", … ""search""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1273, 1273, … 1289]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
72810350,"[1275d 6h 15m 41s 416447µs, 1275d 8h 29m 31s 535127µs, … 1287d 17h 4m 46s 318579µs]","[""fmcg_76376"", ""fmcg_969426"", … ""fmcg_611238""]","[""main"", ""search"", … ""search""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1275, 1275, … 1287]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
180834,"[1288d 13h 47m 34s 80970µs, 1292d 8h 26m 26s 518660µs, … 1303d 12h 3m 11s 641649µs]","[""fmcg_222953"", ""fmcg_855532"", … ""fmcg_989685""]","[""search"", ""search"", … ""i2i""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1288, 1292, … 1303]","[116.628571, 116.628571, … 116.628571]","[4.767532, 4.767532, … 4.767532]","[10.799471, 10.799471, … 10.799471]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"


In [19]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [20]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [21]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})


In [22]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u1""",1.0
"""u2""",0.789998


## Добавляем метрики Precision@k и Recall@k

Пусть:
- $Rel\_u$ — множество релевантных объектов для пользователя $u$ в тесте.
- $Rec\_u@k$ — множество рекомендованных объектов в топ‑k для пользователя $u$.

Тогда для одного пользователя:


$$
Precision@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{k} = \frac{ hits}{k}
$$
то есть, сколько релевантных из рекомендованных.

$$
Recall@k(u) = \frac{|Rec\_u@k \cap Rel\_u|}{|Rel\_u|} = \frac{hits}{Rel\_u}
$$
то есть, сколько релевантных найдено из всех релевантных.

Далее берем среднее по всем пользователям:
$$
Precision@k = \frac{1}{N} \sum_{u=1}^{N} Precision@k(u)
$$
$$
Recall@k = \frac{1}{N} \sum_{u=1}^{N} Recall@k(u)
$$



Precision@k показывает качество самого списка рекомендаций. Она говорит о том, сколько "мусора" мы подсовываем пользователю.

Recall@k показывает охват интересов. Она говорит: "Из всего многообразия товаров, которые этот пользователь мог бы полюбить, какую долю мы реально успели ему показать в нашем коротком топе?".

В рекомендательных системах (особенно когда k маленькое, например 10 или 20) Recall обычно важнее, чем Precision. Precision тоже важна, чтобы не раздражать пользователя нерелевантным мусором, но часто мы готовы пожертвовать точностью (показать пару лишних товаров), лишь бы не упустить то, что пользователь реально ищет (сохранить Recall).

In [23]:
def calculate_metrics(df, k):
    """
    Расчет Precision@k и Recall@k
    df: датафрейм с колонками predicted_items и true_items
    k: количество рекомендаций (TOPK)
    """
    # Обрезаем список предсказаний до k элементов
    top_k_preds = pl.col("predicted_items").list.head(k)
    
    # Ищем пересечение обрезанного списка с правдой
    hits_expr = top_k_preds.list.set_intersection(pl.col("true_items")).list.len()
    
    # Вычисляем метрики одной командой select
    metrics = df.select(
        # Precision = (кол-во попаданий / k), берем среднее по всем юзерам
        (hits_expr / k).mean().alias('precision'),
        
        # Recall = (кол-во попаданий / длину реального списка), берем среднее
        (hits_expr / pl.col('true_items').list.len()).mean().alias('recall')
    )
    
    precision_val = metrics['precision'].item()
    recall_val = metrics['recall'].item()

    return precision_val, recall_val

In [24]:
def calculate_regression_metrics_vectorized(
    test_data,
    user_features: np.ndarray,
    item_features: np.ndarray,
    user_to_idx: dict,
    item_to_idx: dict,
    target_col: str = "log_target",
) -> dict:
    
    # Фильтруем только известные пары user-item
    test_filtered = test_data.filter(
        pl.col("user_id").is_in(list(user_to_idx.keys())) &
        pl.col("item_id").is_in(list(item_to_idx.keys()))
    )
    
    # Получаем индексы
    user_ids = test_filtered["user_id"].to_list()
    item_ids = test_filtered["item_id"].to_list()
    true_scores = test_filtered[target_col].to_numpy()
    
    user_idxs = np.array([user_to_idx[uid] for uid in user_ids])
    item_idxs = np.array([item_to_idx[iid] for iid in item_ids])
    
    # Вычисление скоров
    # pred_score[i] = user_features[user_idx[i]] @ item_features[:, item_idx[i]]
    pred_scores = np.sum(
        user_features[user_idxs] * item_features[:, item_idxs].T, 
        axis=1
    )
    
    # Метрики
    errors = true_scores - pred_scores
    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))
    
    return {
        "rmse": rmse,
        "mae": mae,
        "n_samples": len(true_scores),
    }

Комментарий к коду выше:

* Результат user_predictions — это вектор, где в каждой ячейке лежит число, показывающее "силу связи" (значимость) между этим пользователем и конкретным товаром. Чем больше число, тем сильнее мы хотим порекомендовать этот товар.

* ind_sorted это массив индексов товаров. Индексы идут в порядке убывания скора товара. То есть ind_sorted[0] — это индекс самого подходящего товара, ind_sorted[1] — второго по крутости и т.д.

* top_k_item_ids это оригинальные ID товаров

* top_k_scores это те значимости (скоры), которые мы посчитали (user_predictions). Мы достаем их из большого массива user_predictions по нужным индексам.

Замечание: в функции выше мы исключаем товары, с которыми пользователь взаимодействовал. Скорее всего это не совсем правильный подход. Может быть, стоит исключать только те, которые были просмотренные, а остальные оставлять. 

## EASE

У меня падали все ноутбуки по памяти, поэтому ограничиваю данные 5000 популярных товаров

In [30]:
top_items = (
    train
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .head(5000)
    .collect(engine="streaming") 
)

train = (
    train
    .join(top_items.lazy(), on="item_id")
    .collect(engine="streaming")
)

test = (
    test
    .join(top_items.lazy(), on="item_id")
    .collect(engine="streaming")
)

 # Описание бейзлайна

Будем реализовывать EASE (Embarrassingly Shallow Autoencoder) с коэффициентом регуляризации `REG = 300` и выбирать `TOP_K = 15` рекомендаций

1. Переводим `user_id` и `item_id` в числовые индексы`user_to_idx`, `item_to_idx`
2. Строим разреженную матрицу X (users × items) где:
- строки — пользователи  
- столбцы — товары  
- значения — факт взаимодействия (1)
3. Строим матрицу совместной встречаемости $ G = X^T X $, где:
- $G$ — матрица `(items × items)`, $G_{i,j}$ — количество пользователей, которые взаимодействовали и с товаром $i$, и с товаром $j$
4. Добавляем регуляризацию, чтобы избежать переобучения $ G = G + \lambda I $
5. Получаем матрицу весов по формулам ниже
$$ P = G^{-1} $$
$$ B = -\frac{P}{\text{diag}(P)} $$
$$ B_{i,i} = 0 $$
$B$ — это item-item веса. $B_{i,j}$ показывает, насколько товар $j$ помогает предсказать товар $i$


In [26]:
class EASE:
    def __init__(self, reg: float = 300.0):
        self.reg = reg

    def fit(self, df: pl.DataFrame):
        # индексы
        users = df["user_id"].unique().to_list()
        items = df["item_id"].unique().to_list()

        self.user_to_idx = {u: i for i, u in enumerate(users)}
        self.item_to_idx = {i: j for j, i in enumerate(items)}
        self.idx_to_item = {v: k for k, v in self.item_to_idx.items()}

        # sparse матрица
        user_idx = df["user_id"].replace(self.user_to_idx).to_numpy()
        item_idx = df["item_id"].replace(self.item_to_idx).to_numpy()

        data = np.ones(len(df), dtype=np.float32)
        
        X = coo_matrix(
            (data, (user_idx, item_idx)),
            shape=(len(users), len(items))
        ).tocsr()

        # G = X^T X
        G = (X.T @ X).toarray()

        # регуляризация
        diag_idx = np.diag_indices(G.shape[0])
        G[diag_idx] += self.reg

        # инверсия
        P = np.linalg.inv(G)

        B = -P / np.diag(P)
        B[diag_idx] = 0

        self.W = B
        self.X = X

    def predict(self, user_id: int, k: int = 15):
        if user_id not in self.user_to_idx:
            return []

        u_idx = self.user_to_idx[user_id]

        scores = self.X[u_idx] @ self.W
        scores = np.asarray(scores).ravel()

        seen = self.X[u_idx].toarray().ravel() > 0
        scores[seen] = -np.inf

        top_idx = np.argsort(-scores)[:k]
        return [self.idx_to_item[i] for i in top_idx]

In [27]:
model = EASE(reg=300)
model.fit(train)

In [28]:
def build_user_eval_df(model, test_df, k=15):
    users = test_df["user_id"].unique().to_list()
    rows = []

    for u in users:
        user_data = test_df.filter(pl.col("user_id") == u)

        true_items = user_data["item_id"].to_list()
        if len(true_items) == 0:
            continue

        preds = model.predict(u, k)
        if len(preds) == 0:
            continue

        rows.append({
            "user_id": u,
            "true_items": true_items,
            "relevancy": [1.0] * len(true_items),
            "predicted_items": preds,
            "predicted_scores": list(range(len(preds), 0, -1))
        })

    return pl.DataFrame(rows)

In [29]:
user_eval_df = build_user_eval_df(model, test, k=15)

# NDCG
ndcg_df = ndcg_at_k(
    user_eval_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=15,
)

print("Mean NDCG@15:", ndcg_df["ndcg"].mean())

# Precision / Recall
precision, recall = calculate_metrics(user_eval_df, k=15)
print("Precision@15:", precision)
print("Recall@15:", recall)

Mean NDCG@15: 0.2637590104397235
Precision@15: 0.10264443321542867
Recall@15: 0.008566515569329096
